# I051_ATML_Lab8
## Application: Academic Paper Topic Classification

---

### What this notebook does
Trains and compares **three GNN architectures** on a citation graph:

| Model | Key Idea |
|---|---|
| **GCN** | Averages neighbour features with normalisation |
| **GraphSAGE** | Samples + aggregates neighbourhood features |
| **GAT** | Learns *which* neighbours matter via attention |

**Task:** Given the Cora citation network, classify each paper into one of 7 topics.
- A node = a research paper (features = bag-of-words of abstract)
- An edge = a citation between papers
- Label = one of 7 topics (e.g. Neural Networks, Probabilistic Methods...)

**Dataset:** Cora (built into PyTorch Geometric)
- 2,708 nodes · 5,429 edges · 1,433 features · 7 classes
- Auto-downloads (~5 MB) — no manual upload needed
- Standard benchmark for GNN research

**Why GNNs?**  
Traditional ML ignores the graph structure. A paper's topic depends not just on
its own words, but also on what papers it cites. GNNs propagate information
along edges — each node aggregates features from its neighbours.

In [1]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings, json, os
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
torch.manual_seed(42)
np.random.seed(42)

PyTorch 2.10.0+cu128 | Device: cuda


## 1. Dataset — Cora Citation Network
Auto-downloaded from PyTorch Geometric's servers. ~5 MB.

In [3]:
# Auto-download Cora — stored in ./data/Cora/
dataset = Planetoid(root='./data', name='Cora', transform=NormalizeFeatures())
data    = dataset[0].to(device)

CLASS_NAMES = [
    'Case Based', 'Genetic Algorithms', 'Neural Networks',
    'Probabilistic Methods', 'Reinforcement Learning',
    'Rule Learning', 'Theory'
]

print('=' * 50)
print('  CORA DATASET SUMMARY')
print('=' * 50)
print(f'  Nodes (papers)   : {data.num_nodes:,}')
print(f'  Edges (citations): {data.num_edges:,}')
print(f'  Node features    : {data.num_node_features}')
print(f'  Classes          : {dataset.num_classes}')
print(f'  Train nodes      : {data.train_mask.sum().item()}')
print(f'  Val nodes        : {data.val_mask.sum().item()}')
print(f'  Test nodes       : {data.test_mask.sum().item()}')
print('=' * 50)
print(f'  Classes: {CLASS_NAMES}')

Processing...
Done!


  CORA DATASET SUMMARY
  Nodes (papers)   : 2,708
  Edges (citations): 10,556
  Node features    : 1433
  Classes          : 7
  Train nodes      : 140
  Val nodes        : 500
  Test nodes       : 1000
  Classes: ['Case Based', 'Genetic Algorithms', 'Neural Networks', 'Probabilistic Methods', 'Reinforcement Learning', 'Rule Learning', 'Theory']


In [4]:
# Visualise class distribution
labels    = data.y.cpu().numpy()
counts    = np.bincount(labels)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#0d1117')

colors = ['#58a6ff','#3fb950','#ff7b72','#d29922','#bc8cff','#39d353','#ff6b35']

for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')

# Bar chart
axes[0].bar(range(7), counts, color=colors, edgecolor='#30363d')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels([c.replace(' ', '\n') for c in CLASS_NAMES],
                         color='white', fontsize=8)
axes[0].set_ylabel('Node Count', color='white')
axes[0].set_title('Nodes per Class', color='white', fontweight='bold')
for i, v in enumerate(counts):
    axes[0].text(i, v + 5, str(v), ha='center', color='white', fontsize=8)

# Pie chart
axes[1].pie(counts, labels=CLASS_NAMES, colors=colors,
            autopct='%1.1f%%', textprops={'color': 'white', 'fontsize': 8},
            wedgeprops={'edgecolor': '#30363d'})
axes[1].set_title('Class Distribution', color='white', fontweight='bold')

fig.suptitle('Cora Dataset — Class Distribution', color='white',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved class_distribution.png')

Saved class_distribution.png


## 2. Shared Hyperparameters

In [5]:
HIDDEN   = 64       # hidden layer size — small for speed
DROPOUT  = 0.5
LR       = 0.01
EPOCHS   = 200      # GNNs converge fast on Cora
NUM_FEAT = dataset.num_node_features   # 1433
NUM_CLS  = dataset.num_classes         # 7

print(f'Hidden={HIDDEN} | Dropout={DROPOUT} | LR={LR} | Epochs={EPOCHS}')
print(f'Input features={NUM_FEAT} | Output classes={NUM_CLS}')

Hidden=64 | Dropout=0.5 | LR=0.01 | Epochs=200
Input features=1433 | Output classes=7


## 3. Model A — GCN (Graph Convolutional Network)

**Kipf & Welling (2017)** — the foundational GNN paper.

Each layer computes:  
$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

- $\tilde{A} = A + I$ — adjacency + self-loops  
- $\tilde{D}$ — degree matrix normalisation  
- Every neighbour contributes **equally** (no attention)

In [6]:
class GCN(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, dropout):
        super().__init__()
        self.conv1   = GCNConv(in_ch, hidden)
        self.conv2   = GCNConv(hidden, out_ch)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x   # raw logits — CrossEntropyLoss handles softmax

    def embed(self, x, edge_index):
        """Return hidden-layer embeddings (for visualisation)."""
        with torch.no_grad():
            return F.relu(self.conv1(x, edge_index))

## 4. Model B — GraphSAGE (Graph Sample and Aggregate)

**Hamilton et al. (2017)** — scalable inductive learning.

$$h_v^{(l)} = \sigma\left(W \cdot \text{CONCAT}\left(h_v^{(l-1)},\ \text{MEAN}_{u \in \mathcal{N}(v)} h_u^{(l-1)}\right)\right)$$

- Concatenates the node's own features with the mean of its neighbours  
- Can generalise to **unseen nodes** (inductive) — unlike GCN which is transductive

In [7]:
class GraphSAGE(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, dropout):
        super().__init__()
        self.conv1   = SAGEConv(in_ch, hidden, aggr='mean')
        self.conv2   = SAGEConv(hidden, out_ch, aggr='mean')
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def embed(self, x, edge_index):
        with torch.no_grad():
            return F.relu(self.conv1(x, edge_index))

## 5. Model C — GAT (Graph Attention Network)

**Veličković et al. (2018)** — learns *which neighbours matter most*.

$$h_v^{(l)} = \sigma\left(\sum_{u \in \mathcal{N}(v) \cup \{v\}} \alpha_{vu}\, W h_u^{(l-1)}\right)$$

$$\alpha_{vu} = \text{softmax}\left(\text{LeakyReLU}\left(a^T [Wh_v \| Wh_u]\right)\right)$$

- $\alpha_{vu}$ = learned attention weight between node $v$ and neighbour $u$  
- Multi-head attention stabilises training

In [8]:
class GAT(nn.Module):
    def __init__(self, in_ch, hidden, out_ch, dropout, heads=4):
        super().__init__()
        # heads=4: four parallel attention mechanisms
        self.conv1   = GATConv(in_ch,     hidden, heads=heads,   dropout=dropout, concat=True)
        self.conv2   = GATConv(hidden*heads, out_ch, heads=1,    dropout=dropout, concat=False)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)    # ELU works better than ReLU in GAT
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def embed(self, x, edge_index):
        with torch.no_grad():
            x = F.dropout(x, p=0, training=False)
            return F.elu(self.conv1(x, edge_index))

## 6. Training & Evaluation Functions

In [9]:
def train_epoch(model, optimizer, data):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out   = model(data.x, data.edge_index)
    pred  = out.argmax(dim=1)

    train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
    val_acc   = (pred[data.val_mask]   == data.y[data.val_mask]  ).float().mean().item()
    test_acc  = (pred[data.test_mask]  == data.y[data.test_mask] ).float().mean().item()
    val_loss  = F.cross_entropy(out[data.val_mask], data.y[data.val_mask]).item()
    return train_acc, val_acc, test_acc, val_loss


def train_model(model, data, epochs=EPOCHS, lr=LR, name='Model'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_state   = None
    patience_cnt = 0
    PATIENCE     = 30

    for epoch in range(1, epochs + 1):
        loss = train_epoch(model, optimizer, data)
        tr_acc, va_acc, te_acc, va_loss = evaluate(model, data)
        scheduler.step()

        history['train_loss'].append(loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if patience_cnt >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

        if epoch % 50 == 0:
            print(f'  [{name}] Epoch {epoch:3d} | '
                  f'Loss: {loss:.4f} | '
                  f'Train: {tr_acc*100:.1f}% | '
                  f'Val: {va_acc*100:.1f}% | '
                  f'Test: {te_acc*100:.1f}%')

    # Restore best weights
    model.load_state_dict(best_state)
    _, _, final_test_acc, _ = evaluate(model, data)
    print(f'  [{name}] Best Val: {best_val_acc*100:.2f}% | Final Test: {final_test_acc*100:.2f}%')
    return history, final_test_acc


print('✅ Training utilities ready')

✅ Training utilities ready


## 7. Train All Three Models

In [10]:
# Instantiate
gcn   = GCN(NUM_FEAT, HIDDEN, NUM_CLS, DROPOUT).to(device)
sage  = GraphSAGE(NUM_FEAT, HIDDEN, NUM_CLS, DROPOUT).to(device)
gat   = GAT(NUM_FEAT, HIDDEN, NUM_CLS, DROPOUT, heads=4).to(device)

param_counts = {
    'GCN'       : sum(p.numel() for p in gcn.parameters()),
    'GraphSAGE' : sum(p.numel() for p in sage.parameters()),
    'GAT'       : sum(p.numel() for p in gat.parameters()),
}
for name, cnt in param_counts.items():
    print(f'  {name:<12}: {cnt:,} parameters')

  GCN         : 92,231 parameters
  GraphSAGE   : 184,391 parameters
  GAT         : 369,429 parameters


In [11]:
print('\n── Training GCN ─────────────────────────────────────────')
hist_gcn,  acc_gcn  = train_model(gcn,  data, name='GCN')

print('\n── Training GraphSAGE ───────────────────────────────────')
hist_sage, acc_sage = train_model(sage, data, name='SAGE')

print('\n── Training GAT ─────────────────────────────────────────')
hist_gat,  acc_gat  = train_model(gat,  data, name='GAT')

print('\n✅ All models trained')


── Training GCN ─────────────────────────────────────────
  [GCN] Epoch  50 | Loss: 0.4877 | Train: 97.9% | Val: 80.6% | Test: 82.1%
  Early stop at epoch 76
  [GCN] Best Val: 81.20% | Final Test: 82.90%

── Training GraphSAGE ───────────────────────────────────
  Early stop at epoch 50
  [SAGE] Best Val: 79.60% | Final Test: 80.20%

── Training GAT ─────────────────────────────────────────
  [GAT] Epoch  50 | Loss: 0.5581 | Train: 99.3% | Val: 80.6% | Test: 83.0%
  Early stop at epoch 62
  [GAT] Best Val: 82.00% | Final Test: 82.50%

✅ All models trained


## 8. Training Curves Comparison

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

styles = [
    ('GCN',       hist_gcn,  '#58a6ff', '-'),
    ('GraphSAGE', hist_sage, '#ff6b35', '--'),
    ('GAT',       hist_gat,  '#3fb950', ':'),
]

for ax, key, ylabel in zip(axes, ['val_loss', 'val_acc'], ['Validation Loss', 'Validation Accuracy']):
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')
    for name, hist, color, ls in styles:
        ax.plot(hist[key], label=name, color=color, lw=2, ls=ls)
    ax.set_xlabel('Epoch', color='white')
    ax.set_ylabel(ylabel, color='white')
    ax.set_title(ylabel, color='white', fontweight='bold')
    ax.legend(labelcolor='white', facecolor='#161b22')
    ax.grid(True, alpha=0.15)

fig.suptitle('GCN vs GraphSAGE vs GAT — Training Curves', color='white',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved training_curves.png')

Saved training_curves.png


## 9. Model Comparison — Accuracy Bar Chart

In [13]:
model_names = ['GCN', 'GraphSAGE', 'GAT']
test_accs   = [acc_gcn, acc_sage, acc_gat]
val_accs    = [
    max(hist_gcn['val_acc']),
    max(hist_sage['val_acc']),
    max(hist_gat['val_acc']),
]
model_colors = ['#58a6ff', '#ff6b35', '#3fb950']

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#30363d')

x = np.arange(3)
w = 0.35
b1 = ax.bar(x - w/2, [v*100 for v in val_accs],  w, label='Val Acc',  color=model_colors, alpha=0.6, edgecolor='#30363d')
b2 = ax.bar(x + w/2, [v*100 for v in test_accs], w, label='Test Acc', color=model_colors, alpha=1.0, edgecolor='#30363d')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom',
            color='white', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names, color='white', fontsize=11)
ax.set_ylabel('Accuracy (%)', color='white')
ax.set_title('Model Comparison — Validation & Test Accuracy', color='white', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend(labelcolor='white', facecolor='#161b22')
ax.grid(axis='y', alpha=0.15)

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved accuracy_comparison.png')

Saved accuracy_comparison.png


## 10. Confusion Matrix (Best Model)

In [14]:
# Pick the best model by test accuracy
best_idx   = int(np.argmax(test_accs))
best_model = [gcn, sage, gat][best_idx]
best_name  = model_names[best_idx]
print(f'Best model: {best_name} ({test_accs[best_idx]*100:.2f}% test acc)')

best_model.eval()
with torch.no_grad():
    out  = best_model(data.x, data.edge_index)
    pred = out.argmax(dim=1).cpu().numpy()

true_test = data.y[data.test_mask].cpu().numpy()
pred_test = pred[data.test_mask.cpu().numpy()]

cm = confusion_matrix(true_test, pred_test)

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[c[:8] for c in CLASS_NAMES],
    yticklabels=[c[:8] for c in CLASS_NAMES],
    ax=ax, linewidths=0.5, linecolor='#30363d',
    annot_kws={'color': 'white', 'size': 9}
)
ax.set_xlabel('Predicted', color='white')
ax.set_ylabel('True', color='white')
ax.set_title(f'Confusion Matrix — {best_name} (Test Set)', color='white', fontweight='bold')
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved confusion_matrix.png')

Best model: GCN (82.90% test acc)
Saved confusion_matrix.png


## 11. Node Embedding Visualisation (t-SNE)

In [15]:
from sklearn.manifold import TSNE

best_model.eval()
with torch.no_grad():
    embeddings = best_model.embed(data.x, data.edge_index).cpu().numpy()

print(f'Computing t-SNE on {embeddings.shape[0]} nodes × {embeddings.shape[1]} dims...')
tsne   = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=300)
coords = tsne.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#30363d')

ys = data.y.cpu().numpy()
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask = ys == cls_idx
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=color, label=cls_name, s=10, alpha=0.8)

ax.set_title(f't-SNE of {best_name} Node Embeddings', color='white',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1', color='white')
ax.set_ylabel('t-SNE 2', color='white')
leg = ax.legend(labelcolor='white', facecolor='#161b22',
                markerscale=2, fontsize=9)
leg.get_frame().set_edgecolor('#30363d')
ax.grid(True, alpha=0.1)

plt.tight_layout()
plt.savefig('tsne_embeddings.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved tsne_embeddings.png')

Computing t-SNE on 2708 nodes × 64 dims...
Saved tsne_embeddings.png


## 12. Detailed Evaluation Report

In [16]:
print('=' * 68)
print('  GNN MODEL COMPARISON — Cora Node Classification')
print('=' * 68)
print(f'{"Metric":<30} {"GCN":>10} {"GraphSAGE":>12} {"GAT":>10}')
print('-' * 68)
for label, vals in [
    ('Best Val Accuracy',  [max(h['val_acc'])*100 for h in [hist_gcn, hist_sage, hist_gat]]),
    ('Test Accuracy',      [v*100 for v in test_accs]),
    ('Min Val Loss',       [min(h['val_loss']) for h in [hist_gcn, hist_sage, hist_gat]]),
    ('Epochs Trained',     [len(h['train_loss']) for h in [hist_gcn, hist_sage, hist_gat]]),
    ('# Parameters',       list(param_counts.values())),
]:
    fmt = '{:.2f}%' if 'Accuracy' in label else ('{:.4f}' if 'Loss' in label else '{:,}')
    print(f'{label:<30} {fmt.format(vals[0]):>10} {fmt.format(vals[1]):>12} {fmt.format(vals[2]):>10}')
print('=' * 68)
print(f'  Winner: {best_name} ({test_accs[best_idx]*100:.2f}% test accuracy)')
print('=' * 68)

# Per-class report for best model
print(f'\n  Per-class report ({best_name}):')
print(classification_report(true_test, pred_test, target_names=CLASS_NAMES))

  GNN MODEL COMPARISON — Cora Node Classification
Metric                                GCN    GraphSAGE        GAT
--------------------------------------------------------------------
Best Val Accuracy                  81.20%       79.60%     82.00%
Test Accuracy                      82.90%       80.20%     82.50%
Min Val Loss                       0.8934       0.7773     0.7681
Epochs Trained                         76           50         62
# Parameters                       92,231      184,391    369,429
  Winner: GCN (82.90% test accuracy)

  Per-class report (GCN):
                        precision    recall  f1-score   support

            Case Based       0.68      0.79      0.73       130
    Genetic Algorithms       0.74      0.89      0.81        91
       Neural Networks       0.90      0.88      0.89       144
 Probabilistic Methods       0.92      0.82      0.87       319
Reinforcement Learning       0.85      0.84      0.84       149
         Rule Learning       0.84   

## 13. Save Model Artifacts

In [17]:
os.makedirs('model_artifacts', exist_ok=True)

torch.save(gcn.state_dict(),  'model_artifacts/gcn.pt')
torch.save(sage.state_dict(), 'model_artifacts/sage.pt')
torch.save(gat.state_dict(),  'model_artifacts/gat.pt')

config = {
    'NUM_FEAT'    : NUM_FEAT,
    'NUM_CLS'     : NUM_CLS,
    'HIDDEN'      : HIDDEN,
    'DROPOUT'     : DROPOUT,
    'CLASS_NAMES' : CLASS_NAMES,
    'metrics': {
        'GCN': {
            'test_acc'    : round(acc_gcn, 4),
            'best_val_acc': round(max(hist_gcn['val_acc']), 4),
            'min_val_loss': round(min(hist_gcn['val_loss']), 4),
            'epochs'      : len(hist_gcn['train_loss']),
            'params'      : param_counts['GCN'],
        },
        'GraphSAGE': {
            'test_acc'    : round(acc_sage, 4),
            'best_val_acc': round(max(hist_sage['val_acc']), 4),
            'min_val_loss': round(min(hist_sage['val_loss']), 4),
            'epochs'      : len(hist_sage['train_loss']),
            'params'      : param_counts['GraphSAGE'],
        },
        'GAT': {
            'test_acc'    : round(acc_gat, 4),
            'best_val_acc': round(max(hist_gat['val_acc']), 4),
            'min_val_loss': round(min(hist_gat['val_loss']), 4),
            'epochs'      : len(hist_gat['train_loss']),
            'params'      : param_counts['GAT'],
        },
    }
}
with open('model_artifacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Saved model_artifacts/')
for fn in ['gcn.pt', 'sage.pt', 'gat.pt', 'config.json']:
    print(f'   {fn}')

✅ Saved model_artifacts/
   gcn.pt
   sage.pt
   gat.pt
   config.json


In [19]:

# Step 1: Install Streamlit and pyngrok
!pip install -q streamlit pyngrok

# Step 2: Write app_gnn.py to Colab filesystem
APP_CODE = r'''
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import matplotlib, json, os
matplotlib.use("Agg")

st.set_page_config(page_title="GNN · Paper Classifier", page_icon="🔬",
                   layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    .stApp{background-color:#0d1117}
    h1,h2,h3{color:#58a6ff!important}
    p,li,td,th{color:#c9d1d9}
    .card{background:#161b22;border:1px solid #30363d;border-radius:10px;padding:16px 20px;margin:6px 0;text-align:center}
    .card-val{font-size:1.85rem;font-weight:700}
    .card-lbl{font-size:.78rem;color:#8b949e;margin-top:3px}
    .gcn-tag{background:#001d3d;color:#58a6ff;border-radius:4px;padding:2px 9px;font-size:.8rem;font-weight:700}
    .sage-tag{background:#2d1a00;color:#ff6b35;border-radius:4px;padding:2px 9px;font-size:.8rem;font-weight:700}
    .gat-tag{background:#0d2d0d;color:#3fb950;border-radius:4px;padding:2px 9px;font-size:.8rem;font-weight:700}
    .result-box{background:#161b22;border-radius:8px;padding:16px 20px;margin:8px 0;border:1px solid #30363d}
    .topic-badge{border-radius:20px;padding:5px 16px;font-weight:700;font-size:1rem;display:inline-block;margin:4px 2px}
    .stTextArea textarea{background:#161b22!important;color:#e6edf3!important;border:1px solid #30363d!important}
    .stButton>button{background:linear-gradient(90deg,#1f6feb,#238636);color:white;border:none;font-weight:700;border-radius:8px;padding:10px 28px;font-size:1rem;width:100%}
    .sidebar-info{font-size:.82rem;color:#8b949e;line-height:1.7}
</style>
""", unsafe_allow_html=True)

CLASS_NAMES  = ['Case Based','Genetic Algorithms','Neural Networks','Probabilistic Methods','Reinforcement Learning','Rule Learning','Theory']
CLASS_COLORS = ['#58a6ff','#3fb950','#ff7b72','#d29922','#bc8cff','#39d353','#ff6b35']
CLASS_EMOJIS = ['📚','🧬','🧠','🎲','🤖','📋','📐']

@st.cache_resource(show_spinner="Loading GNN models...")
def load_models():
    try:
        import torch, torch.nn as nn, torch.nn.functional as F
        from torch_geometric.nn import GCNConv, SAGEConv, GATConv
        from torch_geometric.datasets import Planetoid
        from torch_geometric.transforms import NormalizeFeatures
    except ImportError:
        return None
    A = "model_artifacts"
    if not os.path.exists(f"{A}/gcn.pt"): return None
    with open(f"{A}/config.json") as f: cfg = json.load(f)
    NF,NC,H,D = cfg["NUM_FEAT"],cfg["NUM_CLS"],cfg["HIDDEN"],cfg["DROPOUT"]

    class GCN(nn.Module):
        def __init__(self):
            super().__init__(); self.conv1=GCNConv(NF,H); self.conv2=GCNConv(H,NC)
        def forward(self,x,ei): x=F.relu(self.conv1(x,ei)); x=F.dropout(x,p=D,training=self.training); return self.conv2(x,ei)
        def embed(self,x,ei):
            with torch.no_grad(): return F.relu(self.conv1(x,ei))

    class SAGE(nn.Module):
        def __init__(self):
            super().__init__(); self.conv1=SAGEConv(NF,H,aggr="mean"); self.conv2=SAGEConv(H,NC,aggr="mean")
        def forward(self,x,ei): x=F.relu(self.conv1(x,ei)); x=F.dropout(x,p=D,training=self.training); return self.conv2(x,ei)
        def embed(self,x,ei):
            with torch.no_grad(): return F.relu(self.conv1(x,ei))

    class GAT(nn.Module):
        def __init__(self):
            super().__init__(); self.conv1=GATConv(NF,H,heads=4,dropout=D,concat=True); self.conv2=GATConv(H*4,NC,heads=1,dropout=D,concat=False)
        def forward(self,x,ei): x=F.dropout(x,p=D,training=self.training); x=F.elu(self.conv1(x,ei)); x=F.dropout(x,p=D,training=self.training); return self.conv2(x,ei)
        def embed(self,x,ei):
            with torch.no_grad(): x=F.dropout(x,p=0,training=False); return F.elu(self.conv1(x,ei))

    dev = torch.device("cpu")
    gcn=GCN(); gcn.load_state_dict(torch.load(f"{A}/gcn.pt",map_location=dev)); gcn.eval()
    sage=SAGE(); sage.load_state_dict(torch.load(f"{A}/sage.pt",map_location=dev)); sage.eval()
    gat=GAT(); gat.load_state_dict(torch.load(f"{A}/gat.pt",map_location=dev)); gat.eval()
    data = Planetoid(root="./data",name="Cora",transform=NormalizeFeatures())[0].to(dev)
    return {"gcn":gcn,"sage":sage,"gat":gat,"data":data,"cfg":cfg,"torch":torch,"F":F}

def predict_node(node_id, b):
    torch=b["torch"]; F=b["F"]; data=b["data"]; res={}
    for name,model in [("GCN",b["gcn"]),("GraphSAGE",b["sage"]),("GAT",b["gat"])]:
        with torch.no_grad():
            logits=model(data.x,data.edge_index)[node_id]
            probs=F.softmax(logits,dim=0).numpy()
        idx=int(np.argmax(probs))
        res[name]={"pred_class":CLASS_NAMES[idx],"pred_idx":idx,"confidence":float(probs[idx])*100,"all_probs":probs.tolist()}
    return res

with st.sidebar:
    st.markdown("## 🔬 GNN Lab")
    st.markdown("---")
    st.markdown("""<div class="sidebar-info">
<b>Dataset:</b> Cora (2,708 nodes)<br>
<b>Models:</b> GCN · GraphSAGE · GAT<br>
<b>Task:</b> Node classification (7 topics)<br><br>
<b>Sample Node IDs:</b><br>0, 42, 100, 500, 1000, 1500, 2000
</div>""", unsafe_allow_html=True)

st.markdown("<div style='text-align:center;padding:10px 0 4px'><h1 style='font-size:2rem;color:#58a6ff;margin:0'>🔬 Academic Paper Topic Classifier</h1><p style='color:#8b949e;margin:6px 0 0;font-size:.9rem'>GCN · GraphSAGE · GAT &nbsp;·&nbsp; Cora Citation Graph</p></div>",unsafe_allow_html=True)
st.markdown("---")

tab1,tab2,tab3,tab4=st.tabs(["🔍  Classify Node","📊  Dashboard","🗺️  Visualisations","📚  Theory"])

with tab1:
    cl,cr=st.columns([1,1.2],gap="large")
    with cl:
        st.markdown("#### Classify a Paper by Node ID")
        st.caption("Enter any node ID 0–2707")
        nid=st.number_input("Node ID",min_value=0,max_value=2707,value=42,step=1)
        btn=st.button("▶  Classify Paper")
    if btn:
        b=load_models()
        if b is None: st.error("Run the notebook first.")
        else:
            res=predict_node(nid,b)
            data=b["data"]
            true_idx=data.y[nid].item(); true_label=CLASS_NAMES[true_idx]; tc=CLASS_COLORS[true_idx]
            with cr:
                st.markdown(f"#### Results for Node {nid}")
                st.markdown(f"<div style='margin-bottom:8px'><b style='color:#8b949e'>True Label:</b> <span class='topic-badge' style='background:{tc}22;color:{tc};border:1px solid {tc}'>{CLASS_EMOJIS[true_idx]} {true_label}</span></div>",unsafe_allow_html=True)
                tags={"GCN":"gcn-tag","GraphSAGE":"sage-tag","GAT":"gat-tag"}
                for mn,r in res.items():
                    ok="✅" if r["pred_class"]==true_label else "❌"; pc=CLASS_COLORS[r["pred_idx"]]
                    st.markdown(f"<div class='result-box'><span class='{tags[mn]}'>{mn}</span> &nbsp;{ok}<br><span class='topic-badge' style='background:{pc}22;color:{pc};border:1px solid {pc};margin-top:6px'>{CLASS_EMOJIS[r['pred_idx']]} {r['pred_class']}</span><span style='color:#8b949e;font-size:.85rem;margin-left:10px'>{r['confidence']:.1f}%</span></div>",unsafe_allow_html=True)
                bm=max(res,key=lambda m:res[m]["confidence"])
                st.markdown(f"#### Probability Breakdown — {bm}")
                probs=res[bm]["all_probs"]
                fig,ax=plt.subplots(figsize=(7,3.5)); fig.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
                ax.tick_params(colors="white")
                for sp in ax.spines.values(): sp.set_color("#30363d")
                si=np.argsort(probs)[::-1]
                bars=ax.barh([CLASS_NAMES[i][:16] for i in si],[probs[i]*100 for i in si],color=[CLASS_COLORS[i] for i in si],edgecolor="#30363d",height=0.6)
                for bar in bars:
                    ax.text(bar.get_width()+.5,bar.get_y()+bar.get_height()/2,f"{bar.get_width():.1f}%",va="center",color="white",fontsize=8)
                ax.set_xlabel("Probability (%)",color="white"); ax.set_xlim(0,108)
                ax.set_title(f"{bm} — Class Probabilities",color="white",fontweight="bold")
                plt.tight_layout(); st.pyplot(fig); plt.close(fig)

with tab2:
    st.markdown("#### Model Performance Dashboard")
    if os.path.exists("model_artifacts/config.json"):
        with open("model_artifacts/config.json") as f: cfg=json.load(f)
        m=cfg.get("metrics",{})
        c1,c2,c3=st.columns(3)
        for col,(mn,clr,tag) in zip([c1,c2,c3],[("GCN","#58a6ff","gcn-tag"),("GraphSAGE","#ff6b35","sage-tag"),("GAT","#3fb950","gat-tag")]):
            acc=m.get(mn,{}).get("test_acc",0)*100; va=m.get(mn,{}).get("best_val_acc",0)*100
            col.markdown(f"<div class='card'><span class='{tag}'>{mn}</span><div class='card-val' style='color:{clr};margin-top:8px'>{acc:.2f}%</div><div class='card-lbl'>Test Accuracy</div><div style='font-size:.78rem;color:#8b949e;margin-top:6px'>Val: {va:.2f}%</div></div>",unsafe_allow_html=True)
        import pandas as pd
        rows=[{"Model":mn,"Test Acc":f"{m.get(mn,{}).get('test_acc',0)*100:.2f}%","Best Val Acc":f"{m.get(mn,{}).get('best_val_acc',0)*100:.2f}%","Min Val Loss":f"{m.get(mn,{}).get('min_val_loss',0):.4f}","Epochs":m.get(mn,{}).get('epochs','—'),"Params":f"{m.get(mn,{}).get('params',0):,}"} for mn in ["GCN","GraphSAGE","GAT"]]
        st.dataframe(pd.DataFrame(rows),use_container_width=True,hide_index=True)
    if os.path.exists("training_curves.png"): st.image("training_curves.png",use_column_width=True)
    if os.path.exists("accuracy_comparison.png"): st.image("accuracy_comparison.png",use_column_width=True)
    if os.path.exists("confusion_matrix.png"): st.image("confusion_matrix.png",use_column_width=True)

with tab3:
    if os.path.exists("tsne_embeddings.png"):
        st.markdown("##### t-SNE of Node Embeddings"); st.image("tsne_embeddings.png",use_column_width=True)
    if os.path.exists("class_distribution.png"):
        st.markdown("##### Class Distribution"); st.image("class_distribution.png",use_column_width=True)
    lc=st.columns(7)
    for col,(n,c,e) in zip(lc,zip(CLASS_NAMES,CLASS_COLORS,CLASS_EMOJIS)):
        col.markdown(f"<div style='background:{c}22;border:1px solid {c};border-radius:8px;padding:8px 6px;text-align:center'><div style='font-size:1.3rem'>{e}</div><div style='font-size:.7rem;color:{c};font-weight:700'>{n}</div></div>",unsafe_allow_html=True)

with tab4:
    st.markdown("""
## GNN Theory

### Message Passing
```
For each node v:
  1. AGGREGATE  neighbour messages M(v)
  2. COMBINE    with own features
  3. UPDATE     to new representation h_v
```

### Model Comparison
| Feature | GCN | GraphSAGE | GAT |
|---|---|---|---|
| Aggregation | Norm avg | Mean+concat | Attention |
| Neighbour weights | Equal | Equal | **Learned** |
| Inductive | ❌ | ✅ | ✅ |
| Interpretable | ❌ | ❌ | ✅ |

### Cora Dataset
- 2,708 nodes (papers) · 5,429 edges (citations)
- 1,433 bag-of-words features · 7 topic classes
- Only 140 labelled training nodes (20 per class)
""")

st.markdown("<div style='text-align:center;color:#30363d;font-size:.75rem;margin-top:30px'>ATML · GNN · Cora · GCN | GraphSAGE | GAT</div>",unsafe_allow_html=True)
'''

with open("app_gnn.py", "w") as f:
    f.write(APP_CODE)
print("✅ app_gnn.py written to Colab filesystem")

# Step 3: Launch via pyngrok tunnel
import threading, time, subprocess
from pyngrok import ngrok, conf

# ── Optional: set your free ngrok token from https://ngrok.com ──────────────
conf.get_default().auth_token = "3CDfr14enzETXzoDoSrPgDOCsOn_5PwmcgwV7ya91AtVdAcPV"
# ────────────────────────────────────────────────────────────────────────────

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app_gnn.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(5)

public_url = ngrok.connect(8501)
print("=" * 60)
print(f"  🚀 GNN Dashboard live at: {public_url}")
print("=" * 60)
print("  Open the URL above in any browser.")
print("  ⚠  Keep this cell running — closing it stops the app.")
print()
print("  Tabs available:")
print("    🔍  Classify Node   — predict topic for any paper (node ID)")
print("    📊  Dashboard       — metrics, accuracy comparison table")
print("    🗺️   Visualisations  — t-SNE embeddings, class distribution")
print("    📚  Theory          — GCN / SAGE / GAT explanation")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.2 MB/s eta 0:00:00
✅ app_gnn.py written to Colab filesystem
  🚀 GNN Dashboard live at: NgrokTunnel: "https://negligee-thinly-left.ngrok-free.dev" -> "http://localhost:8501"
  Open the URL above in any browser.
  ⚠  Keep this cell running — closing it stops the app.

  Tabs available:
    🔍  Classify Node   — predict topic for any paper (node ID)
    📊  Dashboard       — metrics, accuracy comparison table
    🗺️   Visualisations  — t-SNE embeddings, class distribution
    📚  Theory          — GCN / SAGE / GAT explanation
